# Blocked crossings vs Form 71 inventory

Join blocked crossing events to the Form 71 inventory using `Crossing ID`, then compare blocked vs unblocked crossings and inspect density/sparsity.

In [ ]:
from pathlib import Path

import pandas as pd
from IPython.display import display

from EDA.analyze_blocked_crossings import (
    clean_columns,
    clean_ids,
    make_comparison_table,
    read_csv,
    save_bar,
    save_group_boxplot,
    save_histogram,
    summarize_counts,
    summarize_state_density,
    to_numeric,
)

repo_root = Path.cwd().parent if Path.cwd().name == 'EDA' else Path.cwd()
blocked_path = repo_root / 'data' / 'blocked_crossings_2025(Sheet1).csv'
inventory_path = repo_root / 'data' / 'Crossing_Inventory_Data_(Form_71)_-_Current_20260707.csv'
output_dir = repo_root / 'analysis_outputs'
output_dir.mkdir(exist_ok=True)


In [ ]:
blocked = clean_columns(read_csv(blocked_path))
inventory = clean_columns(read_csv(inventory_path, low_memory=False))

blocked['Crossing ID'] = clean_ids(blocked['Crossing ID'])
inventory['Crossing ID'] = clean_ids(inventory['Crossing ID'])

inventory['Revision Date'] = pd.to_datetime(inventory.get('Revision Date'), errors='coerce')
inventory = inventory.sort_values(['Crossing ID', 'Revision Date']).drop_duplicates('Crossing ID', keep='last')

blocked = blocked.dropna(subset=['Crossing ID']).copy()
blocked['Date/Time'] = pd.to_datetime(blocked.get('Date/Time'), errors='coerce')

blocked_counts = blocked.groupby('Crossing ID').size().rename('blocked_event_count').reset_index()
inventory_with_counts = inventory.merge(blocked_counts, on='Crossing ID', how='left')
inventory_with_counts['blocked_event_count'] = inventory_with_counts['blocked_event_count'].fillna(0).astype(int)
inventory_with_counts['is_blocked'] = inventory_with_counts['blocked_event_count'] > 0

blocked_joined = blocked.merge(inventory, on='Crossing ID', how='left', suffixes=('_blocked', '_inventory'))
blocked_joined.to_csv(output_dir / 'blocked_events_joined_to_inventory.csv', index=False)

state_density = summarize_state_density(inventory_with_counts)
comparison_features = [
    'Annual Average Daily Traffic Count',
    'Annual Average Daily Traffic Year',
    'Total Daylight Thru Trains',
    'Total Nighttime Thru Trains',
    'Total Switching Trains',
    'Total Transit Trains',
    'Number Of Main Tracks',
    'Number Of Siding Tracks',
    'Number Of Yard Tracks',
    'Maximum Timetable Speed',
    'Highway Speed Limit',
]
comparison_table = make_comparison_table(inventory_with_counts, 'is_blocked', comparison_features)


In [ ]:
summary = pd.DataFrame([
    ('blocked_events_total', len(blocked)),
    ('blocked_crossing_ids', blocked['Crossing ID'].nunique()),
    ('inventory_crossings_total', len(inventory)),
    ('inventory_crossings_with_blocked_events', int(inventory_with_counts['is_blocked'].sum())),
    ('inventory_share_with_blocked_events_pct', round(inventory_with_counts['is_blocked'].mean() * 100, 4)),
], columns=['metric', 'value'])

display(summary)
display(summarize_counts(inventory_with_counts.loc[inventory_with_counts['is_blocked'], 'blocked_event_count']))
display(state_density.head(10))
display(comparison_table.head(20))


In [ ]:
reason_counts = blocked['Reason'].value_counts(dropna=False)
state_counts = blocked['State'].value_counts(dropna=False)

save_histogram(inventory_with_counts.loc[inventory_with_counts['is_blocked'], 'blocked_event_count'], output_dir / 'blocked_events_per_crossing_histogram.png')
save_bar(reason_counts.head(10), 'Blocked event reasons', 'Reason', 'Count', output_dir / 'blocked_event_reasons.png', rotation=30)
save_bar(state_counts.head(15), 'Blocked events by state', 'State', 'Blocked event count', output_dir / 'blocked_events_by_state.png', rotation=0)
save_group_boxplot(
    to_numeric(inventory_with_counts.loc[inventory_with_counts['is_blocked'], 'Annual Average Daily Traffic Count']),
    to_numeric(inventory_with_counts.loc[~inventory_with_counts['is_blocked'], 'Annual Average Daily Traffic Count']),
    'AADT by crossing status',
    'Annual Average Daily Traffic Count',
    output_dir / 'aadt_blocked_vs_unblocked.png',
    log_scale=True,
)
save_group_boxplot(
    to_numeric(inventory_with_counts.loc[inventory_with_counts['is_blocked'], 'Total Daylight Thru Trains']),
    to_numeric(inventory_with_counts.loc[~inventory_with_counts['is_blocked'], 'Total Daylight Thru Trains']),
    'Daylight train volume by crossing status',
    'Total Daylight Thru Trains',
    output_dir / 'daylight_trains_blocked_vs_unblocked.png',
    log_scale=True,
)
save_bar(
    state_density.head(15).set_index('State Name')['blocked_crossing_share_pct'],
    'Blocked crossing share by state',
    'State',
    'Blocked crossings / total crossings (%)',
    output_dir / 'blocked_crossing_share_by_state.png',
    rotation=0,
)


The key pattern is sparse but heavy-tailed: most affected crossings have only a few blocked events, while a small number dominate the total.